# RadImageNet-VQA — Qwen2-VL + MedVLM-R1 blank-image controls (Kaggle)

Runs the **same 500 stratified RadImageNet-VQA test questions** as the original **Qwen2-VL-2B-Instruct** and **MedVLM-R1** notebooks, separately with **white** and **black** 512 × 512 RGB images.

- **No original ultrasound/radiology-image model inference**: the dataset's actual images are never passed to a model.
- Each model receives **500 varying questions** for **each color**, totaling **2,000 inferences** if both models and both colors are selected.
- Same prompts, sample selection (`seed=42`), answer parser, `do_sample=False`, `max_new_tokens=16`, and original result/score CSV formats.
- Models load **sequentially** to avoid holding both in GPU memory.
- CSVs are saved separately to `/kaggle/working/blank_control_white/` and `/kaggle/working/blank_control_black/` to avoid overwriting your prior results.
- Existing progress is resumed from the matching CSV when using the same retained Kaggle working directory. Download saved outputs before terminating a session if you need them later.

**Kaggle setup:** Enable GPU and Internet. `HF_TOKEN` is optional for public checkpoints. This notebook only writes results for the blank-image experiment; it never reruns the original-image experiment.


## 1. Dependencies and optional Hugging Face login

In [1]:
%pip install -q -U transformers accelerate qwen-vl-utils datasets scikit-learn huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, re, math, json, gc, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix

# Optional Hugging Face authentication (needed if access is gated).
from huggingface_hub import login
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face authentication configured.")
else:
    print("No HF_TOKEN found: public models may still download; gated models require a token.")
assert torch.cuda.is_available(), "Enable GPU in Kaggle Notebook Settings before running inference."
DEVICE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| dtype:", DEVICE_DTYPE)

Hugging Face authentication configured.
GPU: Tesla T4 | dtype: torch.bfloat16


## 2. Controls and output paths

In [3]:
SEED = 42
MAX_NEW_TOKENS = 16
RUN_COLORS = ("white", "black")    # Or ("white",) / ("black",)
RUN_MODELS = ("qwen2vl_2b", "medvlm_r1")  # Only these two models
MODEL_IDS = {
    "qwen2vl_2b": "Qwen/Qwen2-VL-2B-Instruct",
    "medvlm_r1": "JZPeterPan/MedVLM-R1",
}

OUTPUT_ROOT = Path("/kaggle/working" if Path("/kaggle/working").is_dir() else ".")
for color in RUN_COLORS:
    assert color in ("white", "black"), color
    (OUTPUT_ROOT / f"blank_control_{color}").mkdir(parents=True, exist_ok=True)
for tag in RUN_MODELS:
    assert tag in MODEL_IDS, tag

torch.manual_seed(SEED)
print("Models:", RUN_MODELS)
print("Colors:", RUN_COLORS)
print("Outputs:", OUTPUT_ROOT)


Models: ('qwen2vl_2b', 'medvlm_r1')
Colors: ('white', 'black')
Outputs: /kaggle/working


## 3. Load dataset metadata and reproduce the original sampling

In [4]:
ds = load_dataset("raidium/RadImageNet-VQA", "benchmark", split="test")
meta = ds.remove_columns("image").to_pandas()
meta["content_type"] = meta["metadata"].apply(lambda m: m["content_type"])
meta["anatomy"]      = meta["metadata"].apply(lambda m: m["location"])
meta["question_id"]  = meta["metadata"].apply(lambda m: m["question_id"])
meta["idx"] = np.arange(len(meta))

GROUPS = {
    ("closed", "anatomy"):            "anatomy_yesno",
    ("closed", "pathology"):          "abnormality_yesno",
    ("closed", "pathology_specific"): "pathology_yesno",
    ("multiple_choice", "anatomy"):  "anatomy_mcq",
    ("multiple_choice", "pathology"): "pathology_mcq",
}
meta["group"] = [GROUPS.get((q, c)) for q, c in zip(meta.question_type, meta.content_type)]
pool = meta[meta.group.notna()].copy()
pool["answer"] = pool["answer"].str.strip()
pool.loc[pool.question_type == "closed", "answer"] = pool["answer"].str.lower()

closed = pool[pool.question_type == "closed"]
mcq = pool[pool.question_type == "multiple_choice"]
parts = [g.groupby("answer").sample(50, random_state=SEED) for _, g in closed.groupby("group")]
parts += [g.sample(100, random_state=SEED) for _, g in mcq.groupby("group")]
sample = pd.concat(parts).sort_values("idx").reset_index(drop=True)
assert len(sample) == 500 and sample["idx"].is_unique
assert sample.groupby("group").size().eq(100).all()

# IDs only; preserve the source notebooks' sample-file columns.
for color in RUN_COLORS:
    sample[["idx", "question_id", "group"]].to_csv(
        OUTPUT_ROOT / f"blank_control_{color}" / "vqa_sample.csv", index=False
    )

print(len(pool), "questions in pool")
print(len(sample), "evaluation questions")
print(pd.crosstab(sample.group, sample.answer.where(sample.question_type == "closed", "MCQ")))

README.md:   0%|          | 0.00/13.9k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/59 [00:00<?, ?it/s]

benchmark/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  361MB            

benchmark/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/9000 [00:00<?, ? examples/s]

7000 questions in pool
500 evaluation questions
answer             MCQ  no  yes
group                          
abnormality_yesno    0  50   50
anatomy_mcq        100   0    0
anatomy_yesno        0  50   50
pathology_mcq      100   0    0
pathology_yesno      0  50   50


## 4. Original prompts, answer parser, and per-model inference

In [5]:
LETTERS = "ABCD"

# Identical to the four source notebooks.
def build_prompt(row):
    if row["question_type"] == "multiple_choice":
        opts = "\n".join(f"{L}. {c}" for L, c in zip(LETTERS, row["choices"]))
        return ("You are a medical assistant. Look at this image and answer the following multiple-choice "
                "question with ONLY the letter of the correct option (e.g. 'B'), nothing else.\n\n"
                f"Question: {row['question']}\nOptions:\n{opts}")
    return ("You are a medical assistant. Look at this image and answer the following question "
            "with ONLY the single word 'Yes' or 'No'.\n\n"
            f"Question: {row['question']}")

def parse_mcq(output):
    m = re.search(r"\b([A-D])\b", output)
    return m.group(1) if m else "unparsed"

def parse_yesno(output):
    m = re.search(r"\b(yes|no)\b", output.lower())
    return m.group(1) if m else "unparsed"

def parse(row, output):
    m = re.search(r"<answer>(.*?)</answer>", output, re.S | re.I)
    response = m.group(1) if m else output
    return parse_mcq(response) if row["question_type"] == "multiple_choice" else parse_yesno(response)

from transformers import AutoProcessor, Qwen2VLForConditionalGeneration, Qwen2VLConfig
from qwen_vl_utils import process_vision_info
from huggingface_hub import hf_hub_download

def load_one_model(tag):
    model_id = MODEL_IDS[tag]
    if tag == "medvlm_r1":
        # The same MedVLM-R1 config workaround as the original source notebook.
        with open(hf_hub_download(model_id, "config.json")) as f:
            cfg_dict = json.load(f)
        if cfg_dict.get("use_cache") is None:
            cfg_dict["use_cache"] = True
        config = Qwen2VLConfig.from_dict(cfg_dict)
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_id, config=config, torch_dtype=DEVICE_DTYPE,
            device_map="auto", attn_implementation="sdpa")
        try:
            processor = AutoProcessor.from_pretrained(model_id)
        except Exception as e:
            print("MedVLM processor fallback:", type(e).__name__)
            processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")
    elif tag == "qwen2vl_2b":
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_id, torch_dtype=DEVICE_DTYPE, device_map="auto", attn_implementation="sdpa")
        processor = AutoProcessor.from_pretrained(model_id)
    else:
        raise ValueError(f"Unsupported model: {tag}")
    model.eval()
    print("Loaded", model_id, "on", model.device)
    return model, processor


def predict(image, prompt, tag, model, processor, max_new_tokens=MAX_NEW_TOKENS):
    # Preserve the original Qwen/MedVLM chat-template/image-processor path.
    messages = [{"role": "user", "content": [{"type": "image", "image": image},
                                             {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = out[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


def run_one(row, blank_image, tag, model, processor):
    raw = predict(blank_image, build_prompt(row), tag, model, processor)
    return raw, parse(row, raw)


## 5. Original result and score CSV formats

In [6]:
RESULT_COLUMNS = ["idx", "question_id", "group", "anatomy", "question",
                  "true_answer", "predicted_answer", "raw_output"]

def wilson(k, n, z=1.96):
    if n == 0: return (0, 0)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return c - h, c + h


def write_scores(results, out_dir, tag):
    # Identical score CSV columns and semantics to the source notebook.
    results = results.copy()
    results["correct"] = results.true_answer == results.predicted_answer
    n, k = len(results), int(results.correct.sum())
    lo, hi = wilson(k, n)
    print(f"Accuracy: {k}/{n} = {k/n:.3f} (95% CI {lo:.3f}-{hi:.3f})")
    print(f"Parsed: {(results.predicted_answer != 'unparsed').sum()}/{n}")
    by_group = results.groupby("group").correct.agg(["sum", "count", "mean"]).rename(
        columns={"sum": "correct", "count": "n", "mean": "accuracy"})
    print(by_group.round(3))
    print(results.groupby("anatomy").correct.agg(["count", "mean"]).round(3).rename(
        columns={"count": "n", "mean": "accuracy"}))
    yn = results[results.group.str.endswith("yesno")]
    print(classification_report(yn.true_answer, yn.predicted_answer,
                                labels=["yes", "no"], zero_division=0))
    print(pd.DataFrame(confusion_matrix(yn.true_answer, yn.predicted_answer,
                 labels=["yes", "no", "unparsed"]),
                 index=["true yes", "true no", "true unparsed"],
                 columns=["pred yes", "pred no", "pred unparsed"]))
    by_group.to_csv(out_dir / f"vqa_scores_{tag}.csv")
    return {"model": tag, "color": out_dir.name.replace("blank_control_", ""),
            "n": n, "correct": k, "accuracy": k/n}


def run_condition(tag, color, model, processor):
    out_dir = OUTPUT_ROOT / f"blank_control_{color}"
    out_csv = out_dir / f"vqa_results_{tag}.csv"
    if out_csv.exists():
        done = pd.read_csv(out_csv)
        assert set(RESULT_COLUMNS).issubset(done.columns), f"Bad columns in {out_csv}"
        assert done.idx.is_unique, f"Duplicate case IDs in {out_csv}"
        assert set(done.idx).issubset(set(sample.idx)), f"Unexpected case IDs in {out_csv}"
        rows = done[RESULT_COLUMNS].to_dict("records")
        seen = set(done.idx)
    else:
        rows, seen = [], set()

    blank_image = Image.new("RGB", (512, 512),
                            (255, 255, 255) if color == "white" else (0, 0, 0))
    print(f"{tag} / {color}: {len(seen)} existing, {len(sample)-len(seen)} remaining")
    try:
        for _, r in tqdm(sample.iterrows(), total=len(sample), desc=f"{tag}/{color}"):
            if r["idx"] in seen:
                continue
            raw, pred = run_one(r, blank_image, tag, model, processor)
            rows.append({"idx": r["idx"], "question_id": r["question_id"],
                         "group": r["group"], "anatomy": r["anatomy"],
                         "question": r["question"], "true_answer": r["answer"],
                         "predicted_answer": pred, "raw_output": raw})
            if len(rows) % 50 == 0:
                pd.DataFrame(rows, columns=RESULT_COLUMNS).to_csv(out_csv, index=False)
    finally:
        # Save partial progress even if inference is interrupted.
        if rows:
            pd.DataFrame(rows, columns=RESULT_COLUMNS).to_csv(out_csv, index=False)

    results = pd.DataFrame(rows, columns=RESULT_COLUMNS)
    assert len(results) == len(sample) == 500, f"Incomplete: {tag}/{color} has {len(results)} rows"
    return write_scores(results, out_dir, tag)

## 6. Run selected models sequentially, each color on all 500 questions

In [7]:
summary = []
for tag in RUN_MODELS:
    # Do not download/load a model when both conditions are already fully completed.
    pending_colors = []
    for color in RUN_COLORS:
        result_csv = OUTPUT_ROOT / f"blank_control_{color}" / f"vqa_results_{tag}.csv"
        score_csv = OUTPUT_ROOT / f"blank_control_{color}" / f"vqa_scores_{tag}.csv"
        if result_csv.exists() and score_csv.exists():
            old = pd.read_csv(result_csv)
            if len(old) == len(sample) and old.idx.is_unique and set(old.idx) == set(sample.idx):
                print(f"Skipping completed {tag}/{color}")
                continue
        pending_colors.append(color)
    if not pending_colors:
        continue

    print(f"\n========== MODEL: {tag} ========== ")
    model = processor = None
    try:
        model, processor = load_one_model(tag)
        for color in pending_colors:
            item = run_condition(tag, color, model, processor)
            summary.append(item)
    finally:
        # Keep peak GPU memory low between model checkpoints.
        del model, processor
        gc.collect()
        torch.cuda.empty_cache()
        if hasattr(torch.cuda, "ipc_collect"):
            torch.cuda.ipc_collect()
        print("GPU memory released for", tag)

print("All selected runs finished.")
if summary:
    display(pd.DataFrame(summary))


========== MODEL: qwen2vl_2b ========== 


config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.4k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

... (21 outputs left out)

Accuracy: 219/500 = 0.438 (95% CI 0.395-0.482)
Parsed: 500/500
                   correct    n  accuracy
group                                    
abnormality_yesno       50  100      0.50
anatomy_mcq             36  100      0.36
anatomy_yesno           57  100      0.57
pathology_mcq           25  100      0.25
pathology_yesno         51  100      0.51
              n  accuracy
anatomy                  
abdomen     163     0.589
ankle foot   52     0.327
brain        31     0.387
hip          38     0.395
knee         48     0.271
lung        116     0.388
shoulder     26     0.423
spine        26     0.385
              precision    recall  f1-score   support

         yes       0.61      0.15      0.24       150
          no       0.52      0.90      0.66       150

    accuracy                           0.53       300
   macro avg       0.56      0.53      0.45       300
weighted avg       0.56      0.53      0.45       300

               pred yes  pred no  pred unparsed
true yes

medvlm_r1/black:   0%|          | 0/500 [00:00<?, ?it/s]

Accuracy: 220/500 = 0.440 (95% CI 0.397-0.484)
Parsed: 500/500
                   correct    n  accuracy
group                                    
abnormality_yesno       50  100      0.50
anatomy_mcq             36  100      0.36
anatomy_yesno           57  100      0.57
pathology_mcq           25  100      0.25
pathology_yesno         52  100      0.52
              n  accuracy
anatomy                  
abdomen     163     0.601
ankle foot   52     0.327
brain        31     0.323
hip          38     0.421
knee         48     0.271
lung        116     0.388
shoulder     26     0.423
spine        26     0.385
              precision    recall  f1-score   support

         yes       0.64      0.14      0.23       150
          no       0.52      0.92      0.66       150

    accuracy                           0.53       300
   macro avg       0.58      0.53      0.45       300
weighted avg       0.58      0.53      0.45       300

               pred yes  pred no  pred unparsed
true yes

,model,color,n,correct,accuracy
0,qwen2vl_2b,white,500,208,0.416
1,qwen2vl_2b,black,500,207,0.414
2,medvlm_r1,white,500,219,0.438
3,medvlm_r1,black,500,220,0.440


## 7. Check saved files and package outputs for Kaggle download

In [8]:
for color in RUN_COLORS:
    folder = OUTPUT_ROOT / f"blank_control_{color}"
    print(f"\n{folder}:")
    for p in sorted(folder.glob("*.csv")):
        if p.name.startswith("vqa_results_"):
            n_rows = len(pd.read_csv(p))
            print(f"  {p.name}: {n_rows} rows")
        else:
            print(f"  {p.name}")

zip_path = OUTPUT_ROOT / "radimagenet_qwen_medvlm_blank_controls_results.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for color in RUN_COLORS:
        for file in sorted((OUTPUT_ROOT / f"blank_control_{color}").glob("*.csv")):
            zf.write(file, arcname=str(Path(f"blank_control_{color}") / file.name))
print("\nDownload the CSV bundle from the Kaggle Output pane:", zip_path)


/kaggle/working/blank_control_white:
  vqa_results_medvlm_r1.csv: 500 rows
  vqa_results_qwen2vl_2b.csv: 500 rows
  vqa_sample.csv
  vqa_scores_medvlm_r1.csv
  vqa_scores_qwen2vl_2b.csv

/kaggle/working/blank_control_black:
  vqa_results_medvlm_r1.csv: 500 rows
  vqa_results_qwen2vl_2b.csv: 500 rows
  vqa_sample.csv
  vqa_scores_medvlm_r1.csv
  vqa_scores_qwen2vl_2b.csv

Download the CSV bundle from the Kaggle Output pane: /kaggle/working/radimagenet_qwen_medvlm_blank_controls_results.zip
